In [1]:
from openai import OpenAI
client = OpenAI()

MODEL = "gpt-4o-mini"

In [2]:
SYSTEM_PROMPT = """You are a friendly travel planning assistant.
 
1. Greet the user and collect at least: destination, trip duration, interests
   (food, museums, hiking, beaches, nightlife, shopping...), and optional
   preferences/constraints (budget, travel partners, dates).
2. Ask short follow-up questions until you have that information. Do not ask
   unnecessary questions once you have enough.
3. If the user's information is contradictory or ambiguous, ask for
   clarification before planning.
4. Then generate a personalized plan with: trip title, trip summary,
   day-by-day itinerary, practical travel tips, and estimated budget.
5. Be creative but keep the itinerary realistic and consistent with the
   user's preferences. Reply in the user's language."""

In [3]:
# Prices in USD per 1M tokens (gpt-4o-mini; check the OpenAI pricing page and update if needed)
PRICE_IN = 0.15
PRICE_OUT = 0.60

usage = {"input": 0, "output": 0}
def usage_summary() -> str:
    cost = usage["input"] / 1e6 * PRICE_IN + usage["output"] / 1e6 * PRICE_OUT
    return (f"**Tokens:** {usage['input']} in / {usage['output']} out "
            f"({usage['input'] + usage['output']} total) | **Est. cost:** ${cost:.5f}")

In [4]:
from pydantic import BaseModel, Field
from typing import Optional

class trip(BaseModel):
    title: str = Field(..., description="Trip title")
    summary: str = Field(..., description="Trip summary")
    itinerary: list[str] = Field(..., description="Day-by-day itinerary")
    tips: str = Field(..., description="Practical travel tips")
    budget: str = Field(..., description="Estimated budget")

class Reply(BaseModel):
    message: str = Field(..., description="Text to show the user (a question, or a short intro to the plan)")
    plan: Optional[trip] = Field(None, description="Fill ONLY when enough info is collected; otherwise null")

In [10]:

# The API is stateless: we store the full history and resend it every call.
history = [{"role": "system", "content": SYSTEM_PROMPT}] 
 
def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    response = client.chat.completions.parse(
        model=MODEL,
        messages=history,
        response_format=Reply,
        max_tokens=2000,
    )
    usage["input"] += response.usage.prompt_tokens
    usage["output"] += response.usage.completion_tokens
    reply = response.choices[0].message.parsed
    history.append({"role": "assistant", "content": response.choices[0].message.content})
    return reply

In [11]:
print(chat("Hello! I'm your travel planning assistant. Where would you like to go on your next trip?"))

message='Great to meet you! Where are you thinking of traveling?' plan=None


In [6]:
def plan_to_markdown(p: trip) -> str:
    days = "\n".join(f"- {d}" for d in p.itinerary)
    return (f"# {p.title}\n\n## Summary\n{p.summary}\n\n## Itinerary\n{days}\n\n"
            f"## Practical Tips\n{p.tips}\n\n## Estimated Budget\n{p.budget}")

In [7]:
import panel as pn
pn.extension()

panels = []  # chat display

inp = pn.widgets.TextInput(placeholder="Enter text here…")
button = pn.widgets.Button(name="Chat!", button_type="primary")

def collect_messages(clicks):
    user_text = inp.value.strip()
    if clicks and user_text:
        inp.value = ""
        reply = chat(user_text)  # uses your history-based chat()
        panels.append(pn.Row("You:", pn.pane.Markdown(user_text)))
        panels.append(pn.Row("Assistant:", pn.pane.Markdown(reply)))
    return pn.Column(*panels, pn.pane.Markdown(usage_summary()))

interactive_conversation = pn.bind(collect_messages, button.param.clicks)

dashboard = pn.Column(
    inp,
    button,
    pn.panel(interactive_conversation, loading_indicator=True, sizing_mode="stretch_width", scroll=True),
)
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'6f53a4e7-ff87-45e3-9d97-40ae43c83b70': {'version…

## With image generation

In [ ]:
import ipywidgets as w
import panel as pn
pn.extension()

inp = w.Text(placeholder="Enter text here…", layout=w.Layout(width="500px"))
button = w.Button(description="Chat!", button_style="primary")
image_button = w.Button(description="Create image", button_style="info")
out = w.Output(layout=w.Layout(height="500px", overflow_y="auto"))

def on_click(_):
    user_text = inp.value.strip()
    if not user_text:
        return
    inp.value = ""
    reply = chat(user_text)
    with out:
        display(pn.pane.Markdown(f"**You:** {user_text}"))
        display(pn.pane.Markdown(f"**Assistant:** {reply.message}"))
        if reply.plan:
            display(pn.pane.Markdown(plan_to_markdown(reply.plan)))
        display(pn.pane.Markdown(usage_summary()))

import base64
def make_image_bytes(plan_text: str):
    p = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content":
            "Write a one-paragraph prompt for a beautiful travel poster "
            "illustration based on this trip plan:\n\n" + plan_text}],
        max_tokens=200,
    )
    img = client.images.generate(
        model="gpt-image-1",
        prompt=p.choices[0].message.content,
        size="1024x1024",
    )
    return base64.b64decode(img.data[0].b64_json)

from IPython.display import Image
def on_image_click(_):
    last = next((m["content"] for m in reversed(history) if m["role"] == "assistant"), None)
    if not last:
        return
    image_button.disabled = True
    image_button.description = "Generating..."
    try:
        with out:
            display(Image(data=make_image_bytes(last), width=500))
    finally:
        image_button.disabled = False
        image_button.description = "Create image"

button.on_click(on_click)
image_button.on_click(on_image_click)
display(inp, w.HBox([button, image_button]), out)

Text(value='', layout=Layout(width='500px'), placeholder='Enter text here…')

Output(layout=Layout(height='500px'))